# 06 — Stability Fixes: Reducing Validation Jumpiness

This notebook tests changes from the `stability-fixes` branch against the
notebook 05 baseline. We rerun **exp2 (Normal vs Pure Tumor)** — the same
experiment that showed ±9pp val accuracy swings — and compare curves.

### Changes on this branch

**Safe determinism fixes (tf_pipeline.py):**
1. `deterministic=True` on training interleave (was `False`)
2. Seeded `_shuffle_batch` via `tf.random.Generator` (was unseeded `tf.random.shuffle`)
3. Removed redundant `.repeat()` in `setup_training_pipeline`

**Architecture fix (architectures.py):**
4. Replaced BatchNorm + per-block Dropout with LayerNorm only (kept final 0.5 dropout)
   — eliminates the BN/Dropout train-vs-inference distribution shift

### What to look for
- Smoother val_loss and val_accuracy curves
- Similar or better final accuracy/AUC compared to notebook 05 exp2

In [ ]:
# === SETUP (same as notebook 05) ===

from pathlib import Path
import os

REPO_URL = 'https://github.com/balintstewart77/camelyon16-pathology.git'
COLAB_REPO_PATH = Path('/content/camelyon16-pathology')

if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')

    if COLAB_REPO_PATH.exists():
        import shutil
        shutil.rmtree(COLAB_REPO_PATH)

    !git clone -b stability-fixes {REPO_URL} {COLAB_REPO_PATH}
    !apt-get install -y openslide-tools > /dev/null 2>&1
    !pip install -q -r {COLAB_REPO_PATH}/requirements.txt

    import sys
    sys.path.insert(0, str(COLAB_REPO_PATH))
    os.chdir(COLAB_REPO_PATH)

candidate_roots = [
    Path('/content/drive/MyDrive/camelyon16_data'),
    Path('/content/drive/MyDrive/new_work/Projects/pathovis_project/data'),
]

DATA_ROOT = next((root for root in candidate_roots if root.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not find the CAMELYON16 dataset.\n\n"
        "Please add the shared Google Drive folder as a shortcut to your Drive:\n"
        "  1. Open the shared link\n"
        "  2. Right-click the folder → 'Add shortcut to Drive'\n"
        "  3. Place it directly under 'My Drive'\n"
        "  4. Name it exactly: camelyon16_data\n"
    )
TRAIN_PATH = DATA_ROOT / 'camelyon16_4class_stain_normalised'
TEST_PATH = DATA_ROOT / 'camelyon16_test_stain_normalised'

for path, name in [(TRAIN_PATH, 'Training dataset'), (TEST_PATH, 'Test dataset')]:
    if not path.exists():
        raise FileNotFoundError(
            f"{name} not found at:\n  {path}\n\n"
            "Dataset root found at:\n"
            f"  {DATA_ROOT}\n\n"
            "Expected subfolders:\n"
            "  - camelyon16_4class_stain_normalised\n"
            "  - camelyon16_test_stain_normalised"
        )

print("Dataset paths verified.")
print(f"TRAIN_PATH = {TRAIN_PATH}")
print(f"TEST_PATH  = {TEST_PATH}")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
tf.random.set_seed(42)

from config import DEFAULT_CONFIG
from src.models import run_binary_experiment

In [ ]:
# Same config as notebook 05
DEFAULT_CONFIG.training.normalise_patches = False
DEFAULT_CONFIG.training.val_max_samples_per_class = 4000

TRAIN_DATASET_PATH = TRAIN_PATH

## Experiment 2: Normal vs Pure Tumor

Notebook 05 baseline: val accuracy swung between 0.75 and 0.84 over 9 epochs.

In [ ]:
exp2_results = run_binary_experiment(
    dataset_path=TRAIN_DATASET_PATH,
    experiment_type=2,
    model_name='subtle',
    epochs=15,
    learning_rate=1e-5
)

print(f"\nValidation Results:")
print(f"  Accuracy: {exp2_results['results']['accuracy']:.1%}")
print(f"  AUC: {exp2_results['results']['auc']:.3f}")

## Compare curves

Plot val_accuracy and val_loss to visually assess smoothness.

In [ ]:
import matplotlib.pyplot as plt

history = exp2_results['history']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle('Stability Fixes — Exp 2: Normal vs Pure Tumor', fontsize=13)
plt.tight_layout()
plt.show()

# Print epoch-by-epoch val accuracy for easy comparison
print("\nVal accuracy per epoch:")
for i, acc in enumerate(history.history['val_accuracy']):
    print(f"  Epoch {i+1}: {acc:.4f}")

val_accs = history.history['val_accuracy']
print(f"\nRange: {min(val_accs):.4f} — {max(val_accs):.4f} (span: {max(val_accs)-min(val_accs):.4f})")